> **Solucion.** Laboratorio de clase 4 (MDP del almacen) resuelto: el MDP se modelo a partir de la imagen del enunciado, y Value Iteration + Policy Iteration convergen a la **misma** politica optima (el `assert` final pasa). Corre de principio a fin con `Restart & Run All`.
>
> Nicolas Rodriguez

# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

UP, DOWN, LEFT, RIGHT = (-1, 0), (1, 0), (0, -1), (0, 1)
# perpendiculares (desviaciones) de cada accion; ambas con la misma probabilidad
PERP = {UP: (LEFT, RIGHT), DOWN: (LEFT, RIGHT), LEFT: (UP, DOWN), RIGHT: (UP, DOWN)}


class WarehouseMDP:
    """MDP del almacen, leido de la imagen del enunciado (5 filas x 6 columnas).

    col:      0        1          2            3        4          5
    fila 0:  START    piso       piso         PARED    piso       ENTREGA +10 (term)
    fila 1:  piso     PARED      resbaloso    piso     -3         piso
    fila 2:  piso     resbaloso  CARGA +2(t)  PARED    piso       piso
    fila 3:  piso     piso       resbaloso    piso     piso       MORTAL -10 (term)
    fila 4:  piso     -3         PARED        piso     piso       piso
    """

    def __init__(self):
        self.height, self.width = 5, 6
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 3), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 2)}
        self.terminal_states = {(0, 5): 10.0, (2, 2): 2.0, (3, 5): -10.0}
        self.danger_states = {(1, 4): -3.0, (4, 1): -3.0}   # NO terminales
        self.living_reward = -1.0
        self.gamma = 0.9
        self.actions = [UP, DOWN, LEFT, RIGHT]

    def is_valid_state(self, state):
        r, c = state
        return 0 <= r < self.height and 0 <= c < self.width and state not in self.walls

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if (r, c) not in self.walls]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def _move(self, state, delta):
        nxt = (state[0] + delta[0], state[1] + delta[1])
        return nxt if self.is_valid_state(nxt) else state   # pared/borde -> se queda

    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]                           # estado absorbente
        p_int, p_dev = (0.6, 0.2) if state in self.slippery_states else (0.9, 0.05)
        left, right = PERP[action]
        outcomes = {}
        for delta, p in [(action, p_int), (left, p_dev), (right, p_dev)]:
            ns = self._move(state, delta)
            outcomes[ns] = outcomes.get(ns, 0.0) + p
        return list(outcomes.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    """sum_{s'} T(s,a,s') V(s')"""
    return sum(p * V[ns] for ns, p in grid.get_transition_probs(state, action))


def _greedy_action(grid, state, V):
    """argmax_a E[V(s')] con desempate deterministico (orden fijo de grid.actions)."""
    best_a, best_q = None, -float("inf")
    for a in grid.actions:
        q = expected_next_value(grid, state, a, V)
        if q > best_q + 1e-12:
            best_a, best_q = a, q
    return best_a


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    """V_{k+1}(s) = R(s) + gamma * max_a sum_s' T(s,a,s') V_k(s').  Devuelve (V, n_iter)."""
    V = {s: 0.0 for s in grid.states()}
    for s in grid.terminal_states:
        V[s] = grid.get_reward(s)                           # V(terminal) = R(terminal)
    non_term = [s for s in grid.states() if not grid.is_terminal(s)]
    for it in range(1, max_iter + 1):
        delta = 0.0
        for s in non_term:
            v_old = V[s]
            V[s] = grid.get_reward(s) + grid.gamma * max(
                expected_next_value(grid, s, a, V) for a in grid.actions)
            delta = max(delta, abs(v_old - V[s]))
        if delta < threshold:
            return V, it
    return V, max_iter


def extract_policy(grid, V):
    """pi*(s) = argmax_a sum_s' T(s,a,s') V(s')  (solo estados no terminales)."""
    return {s: _greedy_action(grid, s, V)
            for s in grid.states() if not grid.is_terminal(s)}


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    """V^pi por iteracion de punto fijo (sin el max, la politica esta fija)."""
    V = {s: 0.0 for s in grid.states()}
    for s in grid.terminal_states:
        V[s] = grid.get_reward(s)
    non_term = [s for s in grid.states() if not grid.is_terminal(s)]
    for _ in range(max_iter):
        delta = 0.0
        for s in non_term:
            v_old = V[s]
            V[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, policy[s], V)
            delta = max(delta, abs(v_old - V[s]))
        if delta < threshold:
            break
    return V


def policy_improvement(grid, V):
    """Politica greedy respecto a V."""
    return {s: _greedy_action(grid, s, V)
            for s in grid.states() if not grid.is_terminal(s)}


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    """1) politica arbitraria  2) evaluar  3) mejorar  4) repetir hasta estabilidad.
    Devuelve (politica, V, historia_de_cambios_por_iteracion)."""
    policy = {s: UP for s in grid.states() if not grid.is_terminal(s)}
    history = []
    for _ in range(max_iter):
        V = policy_evaluation(grid, policy, threshold)
        new_policy = policy_improvement(grid, V)
        changed = sum(1 for s in policy if policy[s] != new_policy[s])
        history.append(changed)
        if changed == 0:
            return new_policy, V, history
        policy = new_policy
    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 14

Valores:
 -2.570 |  -1.667 |  -0.639 |   WALL   |  +7.608 | +10.000
 -2.296 |   WALL   |  +0.574 |  +2.171 |  +3.674 |  +7.608
 -1.345 |  -0.176 |  +2.000 |   WALL   |  +3.760 |  +5.583
 -2.211 |  -1.246 |  -0.094 |  +0.279 |  +1.608 | -10.000
 -3.127 |  -4.346 |   WALL   |  -0.801 |  +0.206 |  -1.344

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  #  |  →  |  ↑ 
 →  |  ↑  |  ↑  |  →  |  ↑  | -10
 ↑  |  ↑  |  #  |  ↑  |  ↑  |  ← 

=== POLICY ITERATION ===
Historia: [16, 6, 3, 0]

Valores:
 -2.570 |  -1.667 |  -0.639 |   WALL   |  +7.608 | +10.000
 -2.296 |   WALL   |  +0.574 |  +2.171 |  +3.674 |  +7.608
 -1.345 |  -0.176 |  +2.000 |   WALL   |  +3.760 |  +5.583
 -2.211 |  -1.246 |  -0.094 |  +0.279 |  +1.608 | -10.000
 -3.127 |  -4.346 |   WALL   |  -0.801 |  +0.206 |  -1.344

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  #  |  →  |  ↑ 
 →  |  ↑  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.
